
# Two-zone CM analysis

This notebook summarizes the capacity-market outputs for the reduced A/B system. It stays inside the `cross_border-CM-2zone` workspace and does not use zone C.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.linestyle': '--',
    'grid.alpha': 0.25,
})

BASE_DIR = Path(r'C:/Users/Administrator/OneDrive/Thesis Code/cross_border-CM-2zone')
RESULTS_DIR = BASE_DIR / 'Results'
FIGURES_DIR = BASE_DIR / 'RO_Settlement' / 'Results' / 'Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SAVE_FIGURES = True

def save_fig(fig, name):
    path = FIGURES_DIR / name
    if SAVE_FIGURES:
        fig.savefig(path, bbox_inches='tight')
        print(f'Saved figure: {path}')

def load_zone(zone):
    path = RESULTS_DIR / f'Scenario_1_EOM_Zone_{zone}_ref.csv'
    df = pd.read_csv(path, sep=';')
    df.columns = [c.strip() for c in df.columns]
    df['Zone'] = zone
    return df

zones = ['A', 'B']
raw = pd.concat([load_zone(z) for z in zones], ignore_index=True)
raw['Timestep'] = raw['Timestep'].astype(int)

summary = raw.groupby('Zone').agg(
    EOM_Price_mean=('EOM_price', 'mean'),
    CM_Price_mean=('CM_price', 'mean'),
    TotalCapOffer_mean=('TotalCapOffer', 'mean'),
    TotalCapDemanded_mean=('TotalCapDemanded', 'mean'),
    TotalLocalCapSupply_mean=('TotalLocalCapSupply', 'mean'),
    NetworkManager_mean=('NetworkManager', 'mean'),
    CapacityManager_mean=('CapacityManager', 'mean'),
).reset_index()

summary['CM_Sign'] = np.where(summary['CapacityManager_mean'] >= 0, 'net importer', 'net exporter')
display(summary)


In [ ]:

    # Figure 1: net CM position by zone
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    colors = ['#D55E00' if v < 0 else '#4C72B0' for v in summary['CapacityManager_mean']]
    bars = ax.bar(summary['Zone'], summary['CapacityManager_mean'], color=colors, width=0.58, edgecolor='none')
    ax.axhline(0, color='black', linewidth=1.0)
    ax.set_ylabel('Average net capacity position [MW]')
    ax.set_title('Two-zone CM balance by zone')
    for bar, val in zip(bars, summary['CapacityManager_mean']):
        ax.annotate(f'{val:,.0f}', (bar.get_x() + bar.get_width() / 2, val),
                    xytext=(0, 5 if val >= 0 else -12), textcoords='offset points',
                    ha='center', va='bottom' if val >= 0 else 'top', fontsize=10)
    ax.text(0.02, 0.96, 'Positive = net importer
Negative = net exporter',
            transform=ax.transAxes, va='top', fontsize=9)
    save_fig(fig, 'cm_net_position_2zone.png')
    plt.show()


In [ ]:

# Figure 2: average offer / demand / local supply by zone
metrics = ['TotalCapOffer_mean', 'TotalCapDemanded_mean', 'TotalLocalCapSupply_mean']
labels = ['Capacity offer', 'Capacity demanded', 'Local capacity supply']
x = np.arange(len(summary['Zone']))
width = 0.24

fig, ax = plt.subplots(figsize=(8.2, 4.4))
palette = ['#4C72B0', '#55A868', '#C44E52']
for i, (metric, label, color) in enumerate(zip(metrics, labels, palette)):
    bars = ax.bar(x + (i - 1) * width, summary[metric], width=width, label=label, color=color)
    for bar in bars:
        val = bar.get_height()
        ax.annotate(f'{val:,.0f}', (bar.get_x() + bar.get_width() / 2, val),
                    xytext=(0, 4), textcoords='offset points', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(summary['Zone'])
ax.set_ylabel('Capacity [MW]')
ax.set_title('Average CM quantities by zone')
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.18))
save_fig(fig, 'cm_offer_demand_supply_2zone.png')
plt.show()



### Reading the figure

The two-zone CM run shows the net capacity position directly for A and B. The bar chart gives the sign of the balance, while the second plot shows the level of offer, demand, and local supply behind that balance.
